In [ ]:
import panel as pn
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np # Added this for chart_4
from scipy import stats
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.ensemble import RandomForestRegressor

from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import mean_absolute_error, accuracy_score

from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings('ignore') # Hides all warnings globally

sns.set_theme(style="whitegrid") # Set this ONCE at the top
plt.rcParams.update({
    'font.family': 'serif',      # Forces all charts to use Serif
    'axes.titlesize': 18,
    'axes.labelsize': 12
})

pn.extension()

# --- 1. DATA PREPARATION ---
df = pd.read_csv('Data/Cleaned/Team01_PYVoyagers_cleaned_data.csv')

# --- 2. CHART GENERATORS ---
def create_plot_pane(fig):
    plt.tight_layout()
    plt.close(fig)
    return pn.pane.Matplotlib(fig, tight=True, format='png', sizing_mode='scale_width')
# Descriptive: 1, 5, 7,8
def chart_1(): #Descriptive: 1 (think about it and delete)
    steps_summary = df.groupby(['gender', 'age'])['steps'].mean().reset_index()
    
    # Matches width of other charts to keep font size consistent
    fig, ax = plt.subplots(figsize=(12, 4), facecolor='white')
    fig.patch.set_linewidth(2)     # Set thickness of the border
    fig.patch.set_edgecolor('black') # Set color of the border
    
    # Seaborn handles the 'side-by-side' bars automatically with 'hue'
    sns.barplot(data=steps_summary, x='age', y='steps', hue='gender', ax=ax, palette='muted')
    
    ax.set_title('Average Steps by Age and Gender', fontsize=16, family='serif')
    ax.set_xlabel('Age', fontsize=10)
    ax.set_ylabel('Average steps', fontsize=10)
    ax.legend(title='Gender')
    return create_plot_pane(fig)

def chart_2():# Descriptive:  5
    HYPO, HYPER = 70, 180
    # Data processing inside function
    df_c2 = df.copy()
    df_c2['hr_zone_bpm'] = pd.cut(df_c2['heart_rate'], bins=[0, 80, 110, 300], labels=['Resting (<80)', 'Moderate (80-110)', 'High (≥110)'])
    df_c2['hr_max'] = 220 - df_c2['age']
    df_c2['hr_pct_max'] = df_c2['heart_rate'] / df_c2['hr_max'] * 100
    df_c2['hr_zone_pct'] = pd.cut(df_c2['hr_pct_max'], bins=[0, 50, 70, 200], labels=['Resting/Light (<50%)', 'Moderate (50-70%)', 'High (≥70%)'])
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.patch.set_linewidth(2)     # Set thickness of the border
    fig.patch.set_edgecolor('black') # Set color of the border
    
    sns.boxplot(data=df_c2.dropna(subset=['hr_zone_bpm']), x='hr_zone_bpm', y='glucose', ax=axes[0], palette='coolwarm', hue='hr_zone_bpm', legend=False)
    axes[0].axhline(HYPO, color='red', linestyle='--')
    axes[0].axhline(HYPER, color='orange', linestyle='--')
    axes[0].set_title('Glucose vs Absolute HR Zone', fontsize=16, family='serif')

    sns.boxplot(data=df_c2.dropna(subset=['hr_zone_pct']), x='hr_zone_pct', y='glucose', ax=axes[1], palette='coolwarm', hue='hr_zone_pct', legend=False)
    axes[1].set_title('Glucose vs %HR max Zone', fontsize=16, family='serif')
    
    return create_plot_pane(fig)

def chart_3(): #Descriptive:  7
    fig,(ax1,ax2) = plt.subplots(1,2,figsize = (14,5))
    fig.patch.set_linewidth(2)     # Set thickness of the border
    fig.patch.set_edgecolor('black') # Set color of the border
    ax1.hist(df['sleep_quality_1_10'].dropna(), bins = 10, color ='skyblue', edgecolor = 'black')
    ax1.set_title("Distribution of sleep quality", fontsize=16)
    ax1.set_xlabel('Quality Score', fontsize=10)
    ax1.set_ylabel('Number of Patients', fontsize=10)
    
    df_c3 = df.copy()
    df_c3['disturb_cat'] = df_c3['sleep_quality_1_10'].apply(lambda x:'Disturbed (Score <=5)' if x<= 5 else 'Not Disturbed (Score >5)')
    distrub_pct = df_c3['disturb_cat'].value_counts(normalize = True) * 100
    ax2.bar(distrub_pct.index.tolist(), distrub_pct.values.tolist(), color = ['lightgreen','salmon'])
    ax2.set_title('Frequency of Sleep Disturbances', fontsize=16, family='serif')
    ax2.set_ylabel('Percentage (%)', fontsize=10)

    ax2.set_ylim(0,100)
    return create_plot_pane(fig)

def chart_4(): #  Descriptive: \
    df['time'] = pd.to_datetime(df['time'])
    df['date'] = df['time'].dt.date
    
    # 2. First, sum up the steps/calories per day for EACH patient
    daily_totals = df.groupby(['patient_id', 'date'])[['steps', 'calories']].sum().reset_index()
    
    # 3. Then, find the "Typical" (Mean) of those daily totals per patient
    typical_daily = daily_totals.groupby('patient_id')[['steps', 'calories']].mean().reset_index()
    
    # 4. Visualization
    fig, ax1 = plt.subplots(figsize=(14, 5))
    fig.patch.set_linewidth(2)     # Set thickness of the border
    fig.patch.set_edgecolor('black') # Set color of the border
    x = np.arange(len(typical_daily))
    width = 0.4
    
    # Plot Typical Daily Steps
    ax1.bar(x - width/2, typical_daily['steps'], width, label='Typical Daily Steps', color='skyblue', edgecolor='black')
    ax1.set_ylabel('Mean Steps per Day', color='skyblue', fontweight='bold', fontsize=10)
    ax1.set_xticks(x)
    ax1.set_xticklabels(typical_daily['patient_id'], rotation=45, ha='right', fontsize=10)
    
    # Plot Typical Daily Calories
    ax2 = ax1.twinx()
    ax2.bar(x + width/2, typical_daily['calories'], width, label='Typical Daily Calories', color='salmon', edgecolor='black')
    ax2.set_ylabel('Mean Calories per Day', color='salmon', fontweight='bold')
    
    # Add threshold lines
    ax1.axhline(5000, color='red', linestyle='--', alpha=0.5, label='Sedentary Threshold')
    ax1.axhline(10000, color='skyblue', linestyle='--', alpha=0.5, label='Active Threshold')
    
    plt.title('Typical Daily Activity Level per Patient (Average of Daily Totals)', fontsize=16, family='serif')
    plt.tight_layout()
    #plt.figure(figsize=(8, 4)) 
    
    
    # 5. Categorized Summary
    def classify(steps):
        if steps < 5000: return 'Sedentary'
        if steps < 7500: return 'Low Active'
        if steps < 10000: return 'Somewhat Active'
        return 'Active'
    return create_plot_pane(fig)


    
    

def chart_5(): # Prescritive 11
    cols = ['glucose', 'calories', 'heart_rate', 'steps', 'average_sleep_duration_hrs', 'sleep_quality_1_10']
    corr_matrix = df[cols].corr()
    
    
    fig, ax = plt.subplots(figsize=(8,5))
    fig.patch.set_linewidth(2)     # Set thickness of the border
    fig.patch.set_edgecolor('black') # Set color of the border
    sns.heatmap(corr_matrix, annot=True, cmap='RdBu_r', center=0, fmt=".3f", linewidths=0.5, ax=ax)
    
    ax.set_title('Biomarker Correlation Heatmap', fontsize=16, family='serif')
    plt.xticks(rotation=45, ha='right')
    return create_plot_pane(fig)



def chart_6():
    fig, ax = plt.subplots(figsize=(8, 5))
    fig.patch.set_linewidth(2)     # Set thickness of the border
    fig.patch.set_edgecolor('black') # Set color of the border
    # FIX: Added ax=ax so it uses the right size
    sns.scatterplot(data=df, x='carb_input', y='calories', ax=ax) 
    ax.set_title('Calories vs Carbohydrate Input', fontsize=16)
    return create_plot_pane(fig)

def chart_7():
    
    sns.set_theme(style="whitegrid")
    
    # NEXT-DAY MEAN GLUCOSE
    next_day_glucose = (
        df.groupby(
            ["patient_id", df["time"].dt.date]
        )["glucose"]
        .mean()
        .reset_index(name="next_day_glucose")
    )
    
    # SLEEP FEATURES
    sleep_map = df[
        [
            "patient_id",
            "average_sleep_duration_hrs"
        ]
    ].drop_duplicates()
    
    # MERGE
    
    sleep_glucose = next_day_glucose.merge(
        sleep_map,
        on="patient_id"
    )
    
    # Remove missing values
    sleep_glucose = sleep_glucose.dropna()
    
    # CORRELATION
    r, p = stats.pearsonr(
        sleep_glucose["average_sleep_duration_hrs"],
        sleep_glucose["next_day_glucose"]
    )
    
    fig, ax = plt.subplots(figsize=(8,5))
    fig.patch.set_linewidth(2)     # Set thickness of the border
    fig.patch.set_edgecolor('black') # Set color of the border
    
    sns.regplot(
        data=sleep_glucose,
        x="average_sleep_duration_hrs",
        y="next_day_glucose",
        scatter_kws={"alpha":0.4},
        line_kws={"color":"red"}
    )
    
    plt.title(f"Sleep Duration vs Next-Day Glucose\n" f"r = {r:.2f}, p = {p:.3f}", fontsize=16, family='serif')
    
    plt.xlabel("Average Sleep Duration (hrs)", fontsize = 10)
    plt.ylabel("Next-Day Mean Glucose (mg/dL)", fontsize = 10)
    return create_plot_pane(fig)

def chart_8():
    # Use a local copy to avoid modifying the main dataframe
    df_c8 = df.copy()
    df_c8["hour"] = df_c8["time"].dt.hour
    df_c8["overnight"] = (df_c8["hour"] >= 22) | (df_c8["hour"] <= 6)
    df_c8["late_night_eating"] = (df_c8["hour"] >= 20) & (df_c8["carb_input"] > 0)
    
    overnight_df = df_c8[df_c8["overnight"]].copy().dropna(subset=["glucose", "late_night_eating"])
    overnight_df["late_night_eating"] = overnight_df["late_night_eating"].map({True: "Late Night Eating", False: "No Late Eating"})

    # VISUALIZATION
    fig, ax = plt.subplots(figsize=(8,5))
    fig.patch.set_linewidth(2)     # Set thickness of the border
    fig.patch.set_edgecolor('black') # Set color of the border
    sns.boxplot(data=overnight_df, x="late_night_eating", y="glucose", color="#4cc9f0", width=0.4, ax=ax, showfliers=False)
    sns.stripplot(data=overnight_df, x="late_night_eating", y="glucose", color="#4361ee", alpha=0.35, size=3, ax=ax)
    
    ax.axhline(70, color="#d62828", linestyle="--", label="Hypo (70)")
    ax.axhline(180, color="#f77f00", linestyle="--", label="Hyper (180)")
    ax.set_title("Overnight Glucose: Impact of Late-Night Eating", fontsize=16, family='serif')
    ax.legend()
    
    return create_plot_pane(fig)
def chart_9():
        # Cal the glucose rise, finding the difference bn current row and previous one, 
    # shift(-1)--moves those values up so that the "rise" is recorded on the row where the food was actually eaten, rather than the following row.
    df['glucose_rise'] = df['glucose'].diff().shift(-1)
    
    # Filter for actual meals (Carbs > 0)
    meal_data = df[df['carb_input'] > 0].copy()
    
    #  Create readable categories (Bins)
    meal_data['Carb_Range'] = pd.cut(meal_data['carb_input'], 
                                     bins=[0, 30, 60, 120], 
                                     labels=['Light (0-30g)', 'Medium (30-60g)', 'Heavy (60g+)'])
    
    
    
    # Using duplicates='drop' to handle the high number of 0-step entries
    meal_data['Activity'] = pd.cut(meal_data['steps'], 
                                   bins=[-1, 5, 20,50, 1000], 
                                   labels=['Sedentary', 'Light Walk','Moderate', 'Active'])
    
    # 4. Create the Average Rise Map
    
    pivot_table = meal_data.pivot_table(values='glucose_rise', 
                                        index='Activity', 
                                        columns='Carb_Range', 
                                        aggfunc='mean',
                                        observed=False)
    # 5. Plotting-------------------------------------------
    
    # BAR CHART-----------------------------
    fig, ax = plt.subplots(figsize=(8,5))
    fig.patch.set_linewidth(2)     # Set thickness of the border
    fig.patch.set_edgecolor('black') # Set color of the border
    sns.barplot(data=meal_data, x='Carb_Range', y='glucose_rise', hue='Activity', palette='viridis')
    plt.title('Average Glucose Rise by Meal Size and Activity', fontsize=16, family='serif')
    plt.ylabel('Avg Glucose Change (mg/dL)', fontsize=10)

    return create_plot_pane(fig)

def chart_10():
       # Calculate the glucose change over a 30-minute window (6 intervals)
    # A negative number indicates a drop
    df['glucose_drop_rate'] = df['glucose'].diff(6) 
    
    # 2. Define the "High Intensity/Low Movement" Risk Group
    # We define 'High HR' as >110 BPM and 'Low Steps' as <100 per 5-min window
    high_hr_low_steps = df[(df['heart_rate'] > 110) & (df['steps'] < 100)].copy()
    normal_conditions = df[(df['heart_rate'] <= 110) | (df['steps'] >= 100)].copy()
    
    # 3. Create Comparison Visualization
    fig, ax = plt.subplots(figsize=(8,5))
    fig.patch.set_linewidth(2)     # Set thickness of the border
    fig.patch.set_edgecolor('black') # Set color of the border
    
    sns.kdeplot(normal_conditions['glucose_drop_rate'], label='Normal Activity', fill=True, color='gray')
    sns.kdeplot(high_hr_low_steps['glucose_drop_rate'], label='Risk: High HR + Low Steps', fill=True, color='red')
    
    plt.axvline(x=0, color='black', linestyle='--')
    plt.title('Glucose Drop Rate: High Heart Rate vs. Normal Activity', fontsize=16, family='serif')
    plt.xlabel('Glucose Change in 30 Mins (mg/dL)', fontsize=10)
    plt.ylabel('Frequency', fontsize=10)
    plt.legend()
    return create_plot_pane(fig)

def chart_11():
    
    def plot_late_night_distribution(df, calorie_limit=10):
    # 1. Prep data (Same logic as before)
        # df = df.copy()
        df['time'] = pd.to_datetime(df['time'])
        df['date'] = df['time'].dt.date
        df['hour'] = df['time'].dt.hour
        
        late_meal_dates = df[(df['hour'] >= 20) & (df['calories'] > calorie_limit)]['date'].unique()
        
        # 1. Correct Indentation
        overnight_data = []
        
        # Ensure 'date' exists (or create it if it doesn't)
        if 'date' not in df.columns:
            df['date'] = df['time'].dt.date
    
        for date in df['date'].unique():
            # Define 11 PM to 7 AM window
            start = pd.Timestamp(date) + pd.Timedelta(hours=23)
            end = start + pd.Timedelta(hours=8)
            
            subset = df[(df['time'] >= start) & (df['time'] <= end)].copy()
            
            if not subset.empty:
                # Check if this date is in your late_meal_dates list
                subset['Night_Type'] = 'Late-Night Meal' if date in late_meal_dates else 'Normal Night'
                overnight_data.append(subset)
    
        if not overnight_data:
            print("No overnight data found.")
            return None
    
        plot_df = pd.concat(overnight_data)
    
        # 2. Plotting Logic
        fig, ax = plt.subplots(figsize=(8, 5))
        fig.patch.set_linewidth(2)     # Set thickness of the border
        fig.patch.set_edgecolor('black') # Set color of the border
        sns.set_style("whitegrid")
        
        # Removed split=True because x and hue are the same
        sns.violinplot(
            data=plot_df, 
            x='Night_Type', 
            y='glucose', 
            hue='Night_Type',
            palette={'Late-Night Meal': '#E69F00', 'Normal Night': '#56B4E9'},
            inner="quart", 
            legend=False
        )
    
        # Individual points for density
        sns.stripplot(data=plot_df, x='Night_Type', y='glucose', color='black', alpha=0.2, size=4)
    
        # Target Zone Overlay
        plt.axhspan(70, 150, color='gray', alpha=0.1, label='Target Zone')
        
        plt.title('Distribution of Overnight Glucose: Late-Night Meal vs Normal', fontsize=16, family='serif')
        plt.ylabel('Glucose Level (mg/dL)', fontsize=10)
        plt.xlabel('Eating Habits', fontsize=10)
        plt.legend()
        plot_late_night_distribution(single_patient_df, calorie_limit=10)
        
    
        
        # Only return if you have this specific function defined
        return create_plot_pane(fig)

    import pandas as pd
    import matplotlib.pyplot as plt
    import seaborn as sns
    # --- 2. GLOBAL DATA PREP (Fixes the NameError) ---
# Define cir_df here, OUTSIDE of any function
cir_df = df[(df["carb_input"] > 0) & (df["bolus_volume_delivered"] > 0)].copy()
cir_df["CIR"] = cir_df["carb_input"] / cir_df["bolus_volume_delivered"]
cir_df = cir_df[cir_df["CIR"].between(1, 60)]

# Add the hr_group column needed for chart_13
cir_df["hr_group"] = pd.cut(
    cir_df["heart_rate"], 
    bins=[0, 70, 100, 300], 
    labels=["Low", "Moderate", "High"]
)


    
def chart_13():
    
    
    # VISUALIZATION 4:
    # HEART RATE vs CIR
    fig, ax = plt.subplots(figsize=(8,5))
    fig.patch.set_linewidth(2)     # Set thickness of the border
    fig.patch.set_edgecolor('black') # Set color of the border
    sns.violinplot(
        data=cir_df,
        x="hr_group",
        y="CIR",
        palette="coolwarm"
    )
    
    plt.title("Stress / Heart Rate Impact on CIR", fontsize=16, family='serif')
    plt.xlabel("Heart Rate Group", fontsize=10)
    plt.ylabel("Carb-to-Insulin Ratio", fontsize=10)
    plt.tight_layout()
    # plt.show()
    return create_plot_pane(fig)
    
def chart_14():
    # df["time"] = pd.to_datetime(df["time"])
    # df = df.sort_values(["patient_id", "time"])
    
    
    # 2. FEATURE ENGINEERING
    # Time of day
    df["hour"] = df["time"].dt.hour
    df["time_of_day"] = df["hour"] / 24.0
    
    # Glucose variability (rolling std)
    df["glucose_variability"] = (
        df.groupby("patient_id")["glucose"]
        .rolling(5)
        .std()
        .reset_index(level=0, drop=True)
    )
    df["glucose_variability"] = df["glucose_variability"].fillna(0)
    
    # Glucose slope
    df["glucose_slope"] = df.groupby("patient_id")["glucose"].diff().fillna(0)
    
    # Safe fallback features
    if "basal_rate" not in df.columns:
        df["basal_rate"] = 0
    if "activity_level" not in df.columns:
        df["activity_level"] = 0
    if "carb_intake" not in df.columns:
        df["carb_intake"] = 0
    
    # 3. CLUSTERING (UNSUPERVISED)
    cluster_features = [
        "time_of_day",
        "basal_rate",
        "glucose_variability",
        "glucose_slope"
    ]
    X_cluster = df[cluster_features]
    kmeans = KMeans(n_clusters=2, random_state=42)
    df["stability_cluster"] = kmeans.fit_predict(X_cluster)
    
    # Interpret clusters:
    # lower variability cluster → stable window
    # 4. SUPERVISED MODEL (PREDICTION)
    features = [
        "time_of_day",
        "basal_rate",
        "glucose_variability",
        "glucose_slope",
        "activity_level",
        "carb_intake"
    ]
    X = df[features]
    y = df["stability_cluster"]
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=0.2,
        random_state=42
    )
    model = RandomForestClassifier(n_estimators=200, random_state=42)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    print(classification_report(y_test, y_pred))
    
    fig, ax = plt.subplots(figsize=(8,5))
    fig.patch.set_linewidth(2)     # Set thickness of the border
    fig.patch.set_edgecolor('black') # Set color of the border
    plt.scatter(df["time_of_day"], df["glucose_variability"], 
                c=df["stability_cluster"], cmap="viridis")
    plt.title("Glucose Stability Window Clusters", fontsize=16, family='serif')
    plt.xlabel("Time of Day", fontsize=10)
    plt.ylabel("Glucose Variability", fontsize=10)
    return create_plot_pane(fig)
    
   

def chart_15():
    # 1. Create a copy to avoid SettingWithCopy warnings
    data = df.copy()
    
    # 2. Ensure time is datetime and sorted
    data["time"] = pd.to_datetime(data["time"])
    
    # FIX: Changed 'participant_id' to 'patient_id' to match your dataset
    id_col = "patient_id" if "patient_id" in data.columns else "participant_id"
    data = data.sort_values([id_col, "time"])

    # 3. Feature Engineering
    data["glucose_slope"] = data.groupby(id_col)["glucose"].diff().fillna(0)
    
    # Map actual CSV columns to feature names
    data["carb_intake"] = data["carb_input"] if "carb_input" in data.columns else 0
    data["insulin_bolus"] = data["bolus_volume_delivered"] if "bolus_volume_delivered" in data.columns else 0
    data["activity_level"] = data["steps"] if "steps" in data.columns else 0
    
    data["hour"] = data["time"].dt.hour
    data["time_of_day"] = data["hour"] / 24.0
    
    # 4. TARGET: glucose spike proxy
    data["future_glucose"] = data.groupby(id_col)["glucose"].shift(-3)
    data["glucose_spike"] = data["future_glucose"] - data["glucose"]
    
    # Drop rows without a future spike
    data = data.dropna(subset=["glucose_spike"])
    
    # 5. MODEL PREP
    features = [
        "glucose",
        "glucose_slope",
        "carb_intake",
        "insulin_bolus",
        "activity_level",
        "time_of_day"
    ]
    
    X = data[features]
    y = data["glucose_spike"]
    
    # Train/Test Split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    # 6. TRAINING (Added n_jobs=-1 for speed)
    model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    # 7. VISUALIZATION
    fig, ax = plt.subplots(figsize=(8,5))
    fig.patch.set_linewidth(2)     # Set thickness of the border
    fig.patch.set_edgecolor('black') # Set color of the border
    # We use ax.plot instead of plt.plot for better Panel integration
    ax.plot(y_test.values[:50], label="Actual Spike", alpha=0.7, color='#1f77b4')
    ax.plot(y_pred[:50], label="Predicted Spike", alpha=0.8, color='#ff7f0e', linestyle='--')
    
    ax.set_title("Glucose Spike Prediction (Next 15 Mins)", fontsize=16, family='serif')
    ax.set_xlabel("Test Samples (Last 50)", fontsize=10)
    ax.set_ylabel("Change in Glucose (mg/dL)", fontsize=10)
    ax.legend()
    
    # Print metrics to console for verification
    print(f"MAE: {mean_absolute_error(y_test, y_pred):.2f} mg/dL")
    
    return create_plot_pane(fig)

def placeholder_chart(title):
    fig, ax = plt.subplots(figsize=(10, 2), facecolor='white')
    ax.text(0.5, 0.5, f"Chart: {title}", ha='center')
    return create_plot_pane(fig)

# --- 3. LAYOUT ---
chart_grid = pn.Column(
    chart_1, 
    chart_2, 
    chart_3, 
    chart_4,
    chart_5,
    chart_6,
    chart_7,
    chart_8,
    chart_9,
    chart_10,
    chart_11,
    chart_13,
    chart_14,
    chart_15,
    sizing_mode='stretch_width',
    styles={
        'gap': '-20px', 
        'border': '2px solid #2F4F4F', 
        'border-radius': '10px', 
        'padding': '15px', 
        'background': 'white'
    }
)

# --- 4. FINAL DASHBOARD ASSEMBLY ---

dashboard = pn.Column(
    pn.Row(
        # The <span> tag helps ensure the HUPA-UCM text isn't treated as a link
        pn.pane.Markdown("#  <span style='color:inherit'>PYVoyagers HUPA-UCM Diabetes Dashboard</span>",disable_anchors=True ),
        styles={
            'background': 'White',      # Changed to white
            'color': '#2F4F4F',          # Changed text to Dark Slate Gray for readability
            'padding': '10px',
            'border': '4px solid #DAA520',
            'border-radius': '10px',
            'text-align': 'center'
        },
        width=800,
        align='center',
        sizing_mode='fixed'
    ),
    pn.Spacer(height=10),
    chart_grid
)

dashboard.show()